# MLflow's Model Registry

In [1]:
import os
from mlflow.tracking import MlflowClient


MLFLOW_TRACKING_URI = os.environ['MLFLOW_TRACKING_URI']

## Interacting with the MLflow tracking server

The `MlflowClient` object allows us to interact with...
- an MLflow Tracking Server that creates and manages experiments and runs.
- an MLflow Registry Server that creates and manages registered models and model versions. 

To instantiate it we need to pass a tracking URI and/or a registry URI

In [2]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

client.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/6', creation_time=1780840268964, experiment_id='6', last_update_time=1780840268964, lifecycle_stage='active', name='my-cool-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1778234374914, experiment_id='5', last_update_time=1778234374914, lifecycle_stage='active', name='nyc-taxi-experiment3', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1777208039466, experiment_id='3', last_update_time=1777208039466, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1776676847399, experiment_id='2', last_update_time=1776676847399, lifecycle_stage='active', name='MLflow Quickstart', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflo

Let's check the latest versions for the experiment with id `3`...

In [3]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='3',
    filter_string="metrics.error < 7",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.error ASC"]
)

In [4]:
for run in runs:
    print(f"run id: {run.info.run_id}, run name: {run.info.run_name}, error: {run.data.metrics['error']:.4f}")

run id: ca6d98c602904d7e9bfee6d08be32d60, run name: lyrical-eel-58, error: 6.3086
run id: 1c1339e1a31847e99c0d611dcc75fc6e, run name: trial_31, error: 6.3086
run id: 029b7a9b0d6643cca701a62a54243600, run name: trial_32, error: 6.3098
run id: e3aa823392b2411e90b120efd9383496, run name: trial_16, error: 6.3129
run id: bdd72c3c34b0493495d853ef73113871, run name: trial_27, error: 6.3155


## Interacting with the Model Registry

In [5]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [6]:
# model_uri = "runs:/b2cc9b7375fd4da298456fc1a802b46d/model"
model_uri = "models:/m-d9d64b47008f4d0bbb2f665f8075bb49"
model_name = "nyc-taxi-regressor2"

mlflow.register_model(model_uri=model_uri, name=model_name)

Successfully registered model 'nyc-taxi-regressor2'.
2026/06/08 21:53:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-regressor2, version 1
Created version '1' of model 'nyc-taxi-regressor2'.


<ModelVersion: aliases=[], creation_timestamp=1780955580264, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1780955580264, metrics=None, model_id=None, name='nyc-taxi-regressor2', params=None, run_id='b2cc9b7375fd4da298456fc1a802b46d', run_link='', source='models:/m-d9d64b47008f4d0bbb2f665f8075bb49', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [7]:
version = 1
client.set_model_version_tag(name=model_name, version=version, key="validation_status", value="approved")
client.set_registered_model_alias(name=model_name, version=version, alias="champion")

In [8]:
# model_uri = "runs:/ca6d98c602904d7e9bfee6d08be32d60/models_mlflow"
model_uri = "models:/m-f4dc893d9a0f4683835886094078e555"
model_name = "nyc-taxi-regressor2"

mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'nyc-taxi-regressor2' already exists. Creating a new version of this model...
2026/06/08 21:53:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-regressor2, version 2
Created version '2' of model 'nyc-taxi-regressor2'.


<ModelVersion: aliases=[], creation_timestamp=1780955580483, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1780955580483, metrics=None, model_id=None, name='nyc-taxi-regressor2', params=None, run_id='ca6d98c602904d7e9bfee6d08be32d60', run_link='', source='models:/m-f4dc893d9a0f4683835886094078e555', status='READY', status_message=None, tags={}, user_id='', version='2', workspace='default'>

In [9]:
version = 2
client.set_model_version_tag(name=model_name, version=version, key="validation_status", value="pending")
client.set_registered_model_alias(name=model_name, version=version, alias="challenger")

## Comparing versions and selecting the new `champion` model

In the last section, we will retrieve models registered in the model registry and compare their performance on an unseen test set. The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:

1. Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2021.
2. Download the `DictVectorizer` that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
3. Preprocess the test set using the `DictVectorizer` so we can properly feed the regressors.
4. Make predictions on the test set using the model versions that currently have `challenger` and `champion` aliases, and compare their performance.
5. Based on the results, update the `champion` model version accordingly.


**Note: the model registry doesn't actually deploy the model to production when you assign `champion` alias, it just assign a label to that model version. You should complement the registry with some CI/CD code that does the actual deployment.**

In [10]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd


def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
        df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(dicts)


def test_model(name, alias, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}@{alias}")
    y_pred = model.predict(X_test)
    return {"error": root_mean_squared_error(y_test, y_pred)}

### Prepare the test data

In [11]:
df = read_dataframe("./data/green_tripdata_2021-03.parquet")

In [12]:
run_id = "ca6d98c602904d7e9bfee6d08be32d60"
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/workspaces/mlops-zoomcamp/02-experiment-tracking/preprocessor'

In [13]:
import pickle

with open("preprocessor/preprocessor.pkl", "rb") as f_in:
    dv = pickle.load(f_in)

In [14]:
X_test = preprocess(df, dv)

In [15]:
target = "duration"
y_test = df[target].values

### Compare `champion` and `challenger`

In [16]:
%time test_model(name=model_name, alias="champion", X_test=X_test, y_test=y_test)

2026/06/08 21:53:02 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 3.12.0, required: mlflow==3.11.1)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


CPU times: user 17.3 s, sys: 229 ms, total: 17.5 s
Wall time: 3.16 s


{'error': 6.258074620019596}

In [17]:
%time test_model(name=model_name, alias="challenger", X_test=X_test, y_test=y_test)

2026/06/08 21:53:04 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 3.12.0, required: mlflow==3.11.1)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


CPU times: user 17 s, sys: 91 ms, total: 17.1 s
Wall time: 2.46 s


{'error': 6.258074620019596}

### Update alias and tags

In [18]:
version = 2
client.set_model_version_tag(name=model_name, version=version, key="validation_status", value="approved")
client.set_registered_model_alias(name=model_name, version=version, alias="champion")
client.delete_registered_model_alias(name=model_name, alias="challenger")